# **EVALUATION**

In [ ]:
# ============================================================
# EVALUATION
# ============================================================

import json, re, random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from tqdm import tqdm
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score
)
import os

ACTIONS    = ["ANSWER", "ASK", "ABSTAIN"]
TEST_FILE  = f"{FT_OUT_DIR}/ft_test_mistral.jsonl"
PLOT_DIR   = f"{GDRIVE_PATH}/eval_plots"
os.makedirs(PLOT_DIR, exist_ok=True)
random.seed(42)


# ── helpers ───────────────────────────────────────────────────
def extract_true_label(sample):
    content = sample["messages"][2]["content"]
    if "<decision>" in content and "</decision>" in content:
        s = content.find("<decision>") + len("<decision>")
        e = content.find("</decision>")
        return content[s:e].strip()
    return "UNKNOWN"


def extract_metadata(sample):
    user_content = sample["messages"][1]["content"]
    raw          = json.dumps(sample).lower()
    src = next((s for s in ["quac","sharc","hotpotqa","contract_nli"]
                if s in raw), "unknown")
    is_mt = "<conversation_history>" in user_content
    return src, is_mt


def predict(sample):
    # inject few-shot system prompt — replace original system message
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        sample["messages"][1],   # keep original user turn
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        input_text, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LEN
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = 400,
            do_sample      = False,
            pad_token_id   = tokenizer.pad_token_id,
            eos_token_id   = tokenizer.eos_token_id,
        )

    gen  = out[0][inputs["input_ids"].shape[1]:]
    resp = tokenizer.decode(gen, skip_special_tokens=True).strip()

    if "<decision>" in resp and "</decision>" in resp:
        s = resp.find("<decision>") + len("<decision>")
        e = resp.find("</decision>")
        return resp[s:e].strip(), resp
    return "UNKNOWN", resp


# ── load test set ─────────────────────────────────────────────
print("Loading test set …")
test_samples = []
with open(TEST_FILE) as f:
    for line in f:
        test_samples.append(json.loads(line))

print(f"  Total test  : {len(test_samples)}")

# balanced sample — 150 per class
by_class = defaultdict(list)
for s in test_samples:
    by_class[extract_true_label(s)].append(s)

EVAL_PER_CLASS = 30
eval_samples   = []
for action in ACTIONS:
    pool = by_class.get(action, [])
    random.shuffle(pool)
    eval_samples.extend(pool[:EVAL_PER_CLASS])

random.shuffle(eval_samples)
print(f"  Evaluating  : {len(eval_samples)} samples "
      f"({EVAL_PER_CLASS} per class)")


# ── run inference ─────────────────────────────────────────────
y_true, y_pred     = [], []
sources, is_mt_arr = [], []

print("\nRunning inference …")
for s in tqdm(eval_samples, desc="Eval"):
    true_label  = extract_true_label(s)
    pred_label, _ = predict(s)
    src, is_mt  = extract_metadata(s)

    y_true.append(true_label)
    y_pred.append(pred_label if pred_label in ACTIONS else "UNKNOWN")
    sources.append(src)
    is_mt_arr.append(is_mt)


# ── metrics ───────────────────────────────────────────────────
print("\n" + "="*60)
print("  RESULTS")
print("="*60)

acc         = sum(t==p for t,p in zip(y_true,y_pred)) / len(y_true)
macro_f1    = f1_score(y_true, y_pred, labels=ACTIONS,
                       average="macro",    zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, labels=ACTIONS,
                       average="weighted", zero_division=0)
per_f1      = f1_score(y_true, y_pred, labels=ACTIONS,
                       average=None,       zero_division=0)
unknown_ct  = sum(1 for p in y_pred if p == "UNKNOWN")

print(f"\n  Accuracy    : {acc*100:.2f}%")
print(f"  Macro F1    : {macro_f1*100:.2f}%")
print(f"  Weighted F1 : {weighted_f1*100:.2f}%")
print(f"  Unparseable : {unknown_ct} ({100*unknown_ct/len(y_pred):.1f}%)")
print(f"\n{classification_report(y_true, y_pred, labels=ACTIONS, zero_division=0)}")

# per-source
src_res = defaultdict(lambda: {"c":0,"t":0})
for t,p,src in zip(y_true, y_pred, sources):
    src_res[src]["t"] += 1
    src_res[src]["c"] += int(t==p)
print("  Per-source accuracy:")
for src, r in sorted(src_res.items()):
    print(f"    {src:15}: {r['c']/max(r['t'],1)*100:.1f}%  "
          f"({r['c']}/{r['t']})")

# multi-turn vs single-turn
mt_c  = sum(t==p for t,p,m in zip(y_true,y_pred,is_mt_arr) if m)
mt_t  = sum(1 for m in is_mt_arr if m)
st_c  = sum(t==p for t,p,m in zip(y_true,y_pred,is_mt_arr) if not m)
st_t  = sum(1 for m in is_mt_arr if not m)
print(f"\n  Multi-turn  : {mt_c/max(mt_t,1)*100:.1f}%  ({mt_c}/{mt_t})")
print(f"  Single-turn : {st_c/max(st_t,1)*100:.1f}%  ({st_c}/{st_t})")


# ── plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor("#1a1a2e")
for ax in axes:
    ax.set_facecolor("#1a1a2e")
    for sp in ax.spines.values():
        sp.set_edgecolor("#444")

# confusion matrix
ax = axes[0]
cm = confusion_matrix(y_true, y_pred, labels=ACTIONS)
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(ACTIONS, color="white")
ax.set_yticks(range(3)); ax.set_yticklabels(ACTIONS, color="white")
ax.set_xlabel("Predicted", color="white")
ax.set_ylabel("True",      color="white")
ax.set_title("Confusion Matrix", color="white", fontsize=13)
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i,j]), ha="center", va="center",
                color="white" if cm[i,j] < cm.max()/2 else "black",
                fontsize=12, fontweight="bold")
plt.colorbar(im, ax=ax)

# per-class F1
ax  = axes[1]
bar_colors = ["#57CC99","#4E9AF1","#FF6B6B"]
bars = ax.bar(ACTIONS, per_f1*100, color=bar_colors, width=0.5)
ax.set_ylim(0, 108)
ax.set_ylabel("F1 Score (%)", color="white")
ax.set_title("Per-class F1 Score", color="white", fontsize=13)
ax.tick_params(colors="white")
for bar, val in zip(bars, per_f1):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height()+1.5,
            f"{val*100:.1f}%", ha="center",
            color="white", fontsize=11, fontweight="bold")
ax.axhline(y=macro_f1*100, color="#FFD166", linestyle="--",
           linewidth=1.5, label=f"Macro F1: {macro_f1*100:.1f}%")
ax.legend(fontsize=9, labelcolor="white",
          facecolor="#2a2a3e", edgecolor="#444")

# per-source accuracy
ax        = axes[2]
src_names = sorted(src_res.keys())
src_accs  = [src_res[s]["c"]/max(src_res[s]["t"],1)*100
             for s in src_names]
src_cols  = ["#F4A261","#9B72CF","#57CC99","#4E9AF1"][:len(src_names)]
bars = ax.bar(src_names, src_accs, color=src_cols, width=0.5)
ax.set_ylim(0, 108)
ax.set_ylabel("Accuracy (%)", color="white")
ax.set_title("Accuracy by Source", color="white", fontsize=13)
ax.tick_params(colors="white", axis="both")
for lbl in ax.get_xticklabels():
    lbl.set_color("white"); lbl.set_fontsize(9)
for bar, val in zip(bars, src_accs):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height()+1.5,
            f"{val:.1f}%", ha="center",
            color="white", fontsize=10, fontweight="bold")
ax.axhline(y=acc*100, color="#FFD166", linestyle="--",
           linewidth=1.5, label=f"Overall: {acc*100:.1f}%")
ax.legend(fontsize=9, labelcolor="white",
          facecolor="#2a2a3e", edgecolor="#444")

plt.suptitle(
    f"Planner Evaluation — Mistral-7B LoRA  |  "
    f"Acc: {acc*100:.1f}%  Macro-F1: {macro_f1*100:.1f}%  n={len(eval_samples)}",
    color="white", fontsize=13, y=1.02
)
plt.tight_layout()
plot_path = f"{PLOT_DIR}/eval_results.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print(f"\n  ✅ Plot → {plot_path}")


# ── save metrics ──────────────────────────────────────────────
metrics = {
    "accuracy"       : round(acc, 4),
    "macro_f1"       : round(macro_f1, 4),
    "weighted_f1"    : round(weighted_f1, 4),
    "per_class_f1"   : {a: round(float(f),4) for a,f in zip(ACTIONS,per_f1)},
    "per_source_acc" : {s: round(r["c"]/max(r["t"],1),4)
                        for s,r in src_res.items()},
    "multiturn_acc"  : round(mt_c/max(mt_t,1), 4),
    "singleturn_acc" : round(st_c/max(st_t,1), 4),
    "unknown_preds"  : unknown_ct,
    "eval_n"         : len(eval_samples),
    "confusion_matrix": cm.tolist(),
}
with open(f"{PLOT_DIR}/eval_metrics.json","w") as f:
    json.dump(metrics, f, indent=2)
print(f"  ✅ Metrics → {PLOT_DIR}/eval_metrics.json")

print("\n" + "="*60)
print("  PAPER SUMMARY")
print("="*60)
print(f"  Accuracy     : {acc*100:.1f}%")
print(f"  Macro F1     : {macro_f1*100:.1f}%")
print(f"  ANSWER  F1   : {per_f1[0]*100:.1f}%")
print(f"  ASK     F1   : {per_f1[1]*100:.1f}%")
print(f"  ABSTAIN F1   : {per_f1[2]*100:.1f}%")
print(f"  Multi-turn   : {mt_c/max(mt_t,1)*100:.1f}%")
print(f"  Single-turn  : {st_c/max(st_t,1)*100:.1f}%")
print("="*60)

# **PLANNER + THREE AGENTS**

In [ ]:
# ================================================================
# PLANNER + THREE AGENTS — Full Pipeline
# Planner  → routes to ANSWER / ASK / ABSTAIN agent
# ANSWER   → RAG-grounded response
# ASK      → focused clarification question
# ABSTAIN  → honest refusal with reason
# ================================================================

import re
import torch
from collections import defaultdict


# ================================================================
# SHARED UTILITIES
# ================================================================

def run_planner(query, known_variables, graph_triples,
                missing_variables, history=None):
    """
    Calls the finetuned planner model.
    Returns: (decision, full_response)
    decision ∈ {"ANSWER", "ASK", "ABSTAIN"}
    """
    # build graph context block
    graph_block = (
        "<graph_context>\n"
        + "\n".join(graph_triples)
        + "\n</graph_context>"
    ) if graph_triples else (
        "<graph_context>\nNo relevant nodes found in knowledge graph.\n</graph_context>"
    )

    missing_block = (
        "<missing_variables>\n"
        + "\n".join(f"- {m}" for m in missing_variables)
        + "\n</missing_variables>"
    ) if missing_variables else "<missing_variables>\nnone\n</missing_variables>"

    history_block = ""
    if history:
        history_block = "<conversation_history>\n"
        for h in history:
            history_block += (
                f"Turn {h['turn_id']} | {h['action']} | "
                f"Q: \"{h['query'][:80]}\" | A: {h['response']}\n"
            )
            if h.get("resolved_variable"):
                history_block += f"  → resolved: '{h['resolved_variable']}'\n"
        history_block += "</conversation_history>\n\n"

    user_content = (
        f"{history_block}"
        f"<query>\n{query}\n</query>\n\n"
        f"<known_variables>\n"
        f"{', '.join(known_variables) if known_variables else 'none identified'}\n"
        f"</known_variables>\n\n"
        f"{graph_block}\n\n"
        f"{missing_block}\n\n"
        f"Search the graph context for relevant nodes and decide the correct action."
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        input_text, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LEN
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens    = 400,
            do_sample         = False,
            pad_token_id      = tokenizer.pad_token_id,
            eos_token_id      = tokenizer.eos_token_id,
            stopping_criteria = stop_criteria,
        )

    resp = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    decision = "ABSTAIN"
    if "<decision>" in resp and "</decision>" in resp:
        s        = resp.find("<decision>") + len("<decision>")
        e        = resp.find("</decision>")
        decision = resp[s:e].strip()

    if decision not in {"ANSWER", "ASK", "ABSTAIN"}:
        decision = "ABSTAIN"

    return decision, resp


# ================================================================
# AGENT 1 — ANSWER AGENT (RAG-grounded)
# ================================================================

def answer_agent(query, history=None,
                 top_k_retrieve=20, top_k_rerank=5,
                 alpha=0.5):
    """
    Called when planner decides ANSWER.
    Uses full enhanced RAG pipeline to retrieve context
    and generate a grounded answer.
    Returns: {"action": "ANSWER", "response": str, "sources": list}
    """

    # ── 1. query rewriting ────────────────────────────────────
    working_query = rewrite_query(query)

    # ── 2. retrieval (hybrid BM25 + dense) ───────────────────
    if is_multihop(query):
        sub_qs     = decompose_query(query)
        all_chunks = []
        per_sub    = max(2, top_k_retrieve // len(sub_qs))
        for sq in sub_qs:
            all_chunks.extend(
                hybrid_retrieve(sq, top_k=per_sub, alpha=alpha)
            )
        # deduplicate
        seen_ids, dedup = set(), []
        for c in all_chunks:
            if c["mg_id"] not in seen_ids:
                seen_ids.add(c["mg_id"])
                dedup.append(c)
        all_chunks = dedup
    else:
        all_chunks = hybrid_retrieve(
            working_query, top_k=top_k_retrieve, alpha=alpha
        )

    # ── 3. rerank ─────────────────────────────────────────────
    reranked = rerank(working_query, all_chunks, top_n=top_k_rerank)

    # ── 4. compress ───────────────────────────────────────────
    compressed = compress_context(working_query, reranked)

    # ── 5. build answer prompt ────────────────────────────────
    ctx_block = ""
    for i, c in enumerate(compressed, 1):
        ctx_block += (
            f"\n[Source {i} | {c['source']} | {c.get('granularity','—')}]\n"
            f"{c['text'][:500]}\n"
        )

    history_block = ""
    if history:
        history_block = "Conversation so far:\n"
        for h in history:
            history_block += (
                f"  User: {h['query']}\n"
                f"  Assistant: {h['response']}\n"
            )
        history_block += "\n"

    answer_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content":
          f"""You are a knowledgeable assistant. Answer the query using ONLY the provided context.
Be concise and factual. If the context supports a clear answer, give it directly.

{history_block}Query: {query}

Context:{ctx_block}

Answer:"""}],
        tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(
        answer_prompt, return_tensors="pt",
        truncation=True, max_length=3072
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = 300,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )

    answer = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    sources = [
        {"source": c["source"], "text_preview": c["text"][:80]}
        for c in compressed
    ]

    return {
        "action"          : "ANSWER",
        "response"        : answer,
        "sources"         : sources,
        "retrieved_chunks": len(compressed),
        "rewritten_query" : working_query,
    }


# ================================================================
# AGENT 2 — ASK AGENT (clarification question)
# ================================================================

def ask_agent(query, missing_variables, known_variables,
              graph_triples, history=None):
    """
    Called when planner decides ASK.
    Generates a focused clarification question grounded
    in what is missing from the graph context.
    Returns: {"action": "ASK", "response": str, "missing": list}
    """

    # extract what's missing from planner context
    missing_str = (
        ", ".join(f"'{m}'" for m in missing_variables[:2])
        if missing_variables else "specific details"
    )

    # extract relevant entities from graph for anchoring
    graph_entities = []
    for t in graph_triples[:5]:
        parts = t.split(" | ")
        if len(parts) == 3 and not parts[0].startswith("?"):
            graph_entities.append(parts[0].strip())
    graph_entities = list(dict.fromkeys(graph_entities))[:3]

    anchor = (
        graph_entities[0] if graph_entities
        else (known_variables[0] if known_variables else None)
    )

    # build clarification generation prompt
    context_hint = (
        f"Known entities in context: {', '.join(graph_entities)}. "
        if graph_entities else ""
    )

    history_block = ""
    if history:
        history_block = "Previous turns:\n"
        for h in history[-3:]:   # only last 3 turns for brevity
            history_block += (
                f"  User: {h['query']}\n"
                f"  Assistant [{h['action']}]: {h['response']}\n"
            )
        history_block += "\n"

    clarify_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content":
          f"""You need to ask ONE focused clarification question to help answer the user's query.

{history_block}User query: {query}
Missing information: {missing_str}
{context_hint}

Rules:
- Ask exactly ONE question ending with ?
- Be specific about what is missing
- Reference the query topic directly
- Do not answer the query

Clarification question:"""}],
        tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(
        clarify_prompt, return_tensors="pt",
        truncation=True, max_length=1024
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = 80,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )

    question = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # ensure it ends with ?
    if question and not question.endswith("?"):
        question = question.rstrip(".") + "?"

    # fallback if generation fails
    if not question or len(question) < 10:
        if anchor and missing_variables:
            question = (
                f"Regarding {anchor}, could you clarify "
                f"{missing_variables[0]}?"
            )
        elif missing_variables:
            question = f"Could you provide more details about {missing_variables[0]}?"
        else:
            question = f"Could you provide more specific details about your query?"

    return {
        "action"   : "ASK",
        "response" : question,
        "missing"  : missing_variables,
        "anchor"   : anchor,
    }


# ================================================================
# AGENT 3 — ABSTAIN AGENT (honest refusal)
# ================================================================

def abstain_agent(query, known_variables, graph_triples,
                  missing_variables, history=None):
    """
    Called when planner decides ABSTAIN.
    Generates an honest, informative refusal explaining
    why the query cannot be answered.
    Returns: {"action": "ABSTAIN", "response": str}
    """

    has_graph    = len(graph_triples) > 0
    has_known    = len(known_variables) > 0
    is_vague     = len(query.split()) < 5

    # determine reason for abstaining
    if is_vague and not has_known:
        reason = "The query is too vague to match any information in the knowledge base."
    elif has_graph and not has_known:
        reason = ("The knowledge base contains related content but "
                  "it does not connect to the specific question asked.")
    elif missing_variables:
        reason = (
            f"The required information — "
            f"{', '.join(missing_variables[:2])} — "
            f"is entirely absent from the knowledge base and "
            f"cannot be obtained through clarification."
        )
    else:
        reason = "The topic of this query is not covered in the available knowledge base."

    history_block = ""
    if history:
        resolved = [
            h["resolved_variable"]
            for h in history
            if h.get("resolved_variable")
        ]
        if resolved:
            history_block = (
                f"Note: The following were already clarified in this conversation: "
                f"{', '.join(resolved)}. "
            )

    abstain_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content":
          f"""You cannot answer the following query from the available knowledge base.
Write a brief, honest, and helpful refusal. Explain why you cannot answer.
Do NOT make up information. Do NOT suggest you might know the answer.
End with a suggestion of what type of source might help.

{history_block}Query: {query}
Reason you cannot answer: {reason}

Your response:"""}],
        tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(
        abstain_prompt, return_tensors="pt",
        truncation=True, max_length=1024
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = 120,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )

    refusal = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # fallback
    if not refusal or len(refusal) < 20:
        refusal = (
            f"I'm unable to answer this query. {reason} "
            f"You may want to consult a specialised source or expert."
        )

    return {
        "action"  : "ABSTAIN",
        "response": refusal,
        "reason"  : reason,
    }


# ================================================================
# MAIN PIPELINE — Planner routes to correct agent
# ================================================================

def run_pipeline(query,
                 known_variables   = None,
                 graph_triples     = None,
                 missing_variables = None,
                 history           = None,
                 kg                = None,
                 verbose           = True):
    """
    Full pipeline:
    1. Retrieve KG context for query
    2. Run planner → get decision
    3. Route to correct agent
    4. Return structured result
    """

    known_variables   = known_variables   or []
    graph_triples     = graph_triples     or []
    missing_variables = missing_variables or []

    # ── auto-retrieve KG context if kg provided ───────────────
    if kg is not None and not graph_triples:
        from sentence_transformers import SentenceTransformer
        retrieved, matched, seeds = get_subgraph_triples(
            query, known_variables, missing_variables, "ASK"
        )
        graph_triples     = cap_requires_edges(retrieved, action="ASK")
        missing_variables = infer_effective_missing(
            query, known_variables, missing_variables,
            graph_triples, "ASK"
        )

    if verbose:
        print(f"\n{'='*60}")
        print(f"  Query    : {query}")
        print(f"  Known    : {known_variables}")
        print(f"  Triples  : {len(graph_triples)}")
        print(f"  Missing  : {missing_variables}")

    # ── 1. planner decision ───────────────────────────────────
    decision, planner_response = run_planner(
        query, known_variables, graph_triples,
        missing_variables, history
    )

    if verbose:
        print(f"  Decision : {decision}")

    # ── 2. route to agent ─────────────────────────────────────
    if decision == "ANSWER":
        result = answer_agent(
            query, history=history
        )

    elif decision == "ASK":
        result = ask_agent(
            query, missing_variables, known_variables,
            graph_triples, history=history
        )

    else:   # ABSTAIN
        result = abstain_agent(
            query, known_variables, graph_triples,
            missing_variables, history=history
        )

    result["planner_reasoning"] = planner_response
    result["graph_triples"]     = graph_triples

    if verbose:
        print(f"  Response : {result['response'][:120]}")
        print(f"{'='*60}")

    return result

In [ ]:
# ================================================================
# SYNTHETIC PIPELINE VERIFICATION — SELF-CONTAINED
# Run this after: model + tokenizer + SYSTEM_PROMPT + stop_criteria
# No external dependencies needed
# ================================================================

import torch, re
from collections import defaultdict

MAX_SEQ_LEN = 1024   # match training

# ================================================================
# LIGHTWEIGHT STUBS — replace full RAG pipeline
# ================================================================

def _llm_generate(prompt_text, max_new_tokens=300):
    inputs = tokenizer(
        prompt_text, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LEN
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            pad_token_id   = tokenizer.pad_token_id,
            eos_token_id   = tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()


def _rewrite_query(query):
    """Lightweight query rewriter."""
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content":
          f"Rewrite this query to be more specific for retrieval. "
          f"Return ONLY the rewritten query.\n\nQuery: {query}\nRewritten:"}],
        tokenize=False, add_generation_prompt=True
    )
    result = _llm_generate(prompt, max_new_tokens=60)
    return result.split("\n")[0].strip() or query


def _simple_retrieve(query, graph_triples, kb_text=""):
    """
    For synthetic cases: just return the pre-built graph triples
    formatted as retrieval chunks. No FAISS needed.
    """
    if not graph_triples and not kb_text:
        return []
    chunks = []
    if kb_text:
        chunks.append({
            "mg_id"      : "syn_chunk_0",
            "granularity": "coarse",
            "source"     : "synthetic",
            "text"       : kb_text,
            "fused_score": 1.0,
        })
    if graph_triples:
        triple_text = "\n".join(graph_triples)
        chunks.append({
            "mg_id"      : "syn_chunk_kg",
            "granularity": "fine",
            "source"     : "kg_triples",
            "text"       : triple_text,
            "fused_score": 0.9,
        })
    return chunks


# ================================================================
# PLANNER
# ================================================================

def run_planner(query, known_variables, graph_triples,
                missing_variables, history=None):
    graph_block = (
        "<graph_context>\n" + "\n".join(graph_triples) + "\n</graph_context>"
    ) if graph_triples else (
        "<graph_context>\nNo relevant nodes found in knowledge graph.\n</graph_context>"
    )

    missing_block = (
        "<missing_variables>\n"
        + "\n".join(f"- {m}" for m in missing_variables)
        + "\n</missing_variables>"
    ) if missing_variables else "<missing_variables>\nnone\n</missing_variables>"

    history_block = ""
    if history:
        history_block = "<conversation_history>\n"
        for h in history:
            history_block += (
                f"Turn {h['turn_id']} | {h['action']} | "
                f"Q: \"{h['query'][:80]}\" | A: {h['response']}\n"
            )
            if h.get("resolved_variable"):
                history_block += f"  → resolved: '{h['resolved_variable']}'\n"
        history_block += "</conversation_history>\n\n"

    user_content = (
        f"{history_block}"
        f"<query>\n{query}\n</query>\n\n"
        f"<known_variables>\n"
        f"{', '.join(known_variables) if known_variables else 'none identified'}\n"
        f"</known_variables>\n\n"
        f"{graph_block}\n\n"
        f"{missing_block}\n\n"
        f"Search the graph context for relevant nodes and decide the correct action."
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        input_text, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LEN
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens    = 400,
            do_sample         = False,
            pad_token_id      = tokenizer.pad_token_id,
            eos_token_id      = tokenizer.eos_token_id,
            stopping_criteria = stop_criteria,
        )

    resp = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    decision = "ABSTAIN"
    if "<decision>" in resp and "</decision>" in resp:
        s        = resp.find("<decision>") + len("<decision>")
        e        = resp.find("</decision>")
        decision = resp[s:e].strip()

    if decision not in {"ANSWER", "ASK", "ABSTAIN"}:
        decision = "ABSTAIN"

    return decision, resp


# ================================================================
# ANSWER AGENT — self-contained
# ================================================================

def answer_agent(query, graph_triples=None,
                 kb_text="", history=None):
    chunks = _simple_retrieve(query, graph_triples or [], kb_text)

    ctx_block = ""
    for i, c in enumerate(chunks, 1):
        ctx_block += (
            f"\n[Source {i} | {c['source']} | {c.get('granularity','—')}]\n"
            f"{c['text'][:500]}\n"
        )

    history_block = ""
    if history:
        history_block = "Conversation so far:\n"
        for h in history[-3:]:
            history_block += (
                f"  User: {h['query']}\n"
                f"  Assistant: {h['response']}\n"
            )
        history_block += "\n"

    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content":
          f"You are a knowledgeable assistant. Answer the query using ONLY "
          f"the provided context. Be concise and factual.\n\n"
          f"{history_block}Query: {query}\n\nContext:{ctx_block}\n\nAnswer:"}],
        tokenize=False, add_generation_prompt=True
    )

    answer = _llm_generate(prompt, max_new_tokens=200)

    return {
        "action"  : "ANSWER",
        "response": answer,
        "sources" : [{"source": c["source"], "text_preview": c["text"][:80]}
                     for c in chunks],
    }


# ================================================================
# ASK AGENT — self-contained
# ================================================================

def ask_agent(query, missing_variables, known_variables,
              graph_triples=None, history=None):

    missing_str = (
        ", ".join(f"'{m}'" for m in missing_variables[:2])
        if missing_variables else "specific details"
    )

    graph_entities = []
    for t in (graph_triples or [])[:5]:
        parts = t.split(" | ")
        if len(parts) == 3 and not parts[0].startswith("?"):
            graph_entities.append(parts[0].strip())
    graph_entities = list(dict.fromkeys(graph_entities))[:3]

    anchor = (
        graph_entities[0] if graph_entities
        else (known_variables[0] if known_variables else None)
    )

    history_block = ""
    if history:
        history_block = "Previous turns:\n"
        for h in history[-3:]:
            history_block += (
                f"  User: {h['query']}\n"
                f"  Assistant [{h['action']}]: {h['response']}\n"
            )
        history_block += "\n"

    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content":
          f"Ask ONE focused clarification question to help answer the user's query.\n\n"
          f"{history_block}User query: {query}\n"
          f"Missing information: {missing_str}\n"
          f"Known context: {', '.join(graph_entities) if graph_entities else 'none'}\n\n"
          f"Rules: Ask exactly ONE question ending with ?\n"
          f"Be specific about what is missing. Reference the query topic.\n\n"
          f"Clarification question:"}],
        tokenize=False, add_generation_prompt=True
    )

    question = _llm_generate(prompt, max_new_tokens=80)
    question = question.split("\n")[0].strip()

    if not question.endswith("?"):
        question = question.rstrip(".") + "?"

    if not question or len(question) < 10:
        if anchor and missing_variables:
            question = f"Regarding {anchor}, could you clarify {missing_variables[0]}?"
        elif missing_variables:
            question = f"Could you provide more details about {missing_variables[0]}?"
        else:
            question = "Could you provide more specific details about your query?"

    return {
        "action"  : "ASK",
        "response": question,
        "missing" : missing_variables,
        "anchor"  : anchor,
    }


# ================================================================
# ABSTAIN AGENT — self-contained
# ================================================================

def abstain_agent(query, known_variables, graph_triples=None,
                  missing_variables=None, history=None):

    has_graph  = len(graph_triples or []) > 0
    has_known  = len(known_variables or []) > 0
    is_vague   = len(query.split()) < 5

    if is_vague and not has_known:
        reason = "The query is too vague to match any information in the knowledge base."
    elif has_graph and not has_known:
        reason = ("The knowledge base contains related content but "
                  "it does not connect to the specific question asked.")
    elif missing_variables:
        reason = (
            f"The required information — "
            f"{', '.join((missing_variables or [])[:2])} — "
            f"is entirely absent from the knowledge base."
        )
    else:
        reason = "The topic of this query is not covered in the available knowledge base."

    history_note = ""
    if history:
        resolved = [h["resolved_variable"] for h in history
                    if h.get("resolved_variable")]
        if resolved:
            history_note = (
                f"Already clarified in this conversation: "
                f"{', '.join(resolved)}. "
            )

    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content":
          f"You cannot answer the following query from the available knowledge base.\n"
          f"Write a brief, honest refusal. Explain why you cannot answer.\n"
          f"Do NOT make up information.\n\n"
          f"{history_note}Query: {query}\n"
          f"Reason: {reason}\n\n"
          f"Your response:"}],
        tokenize=False, add_generation_prompt=True
    )

    refusal = _llm_generate(prompt, max_new_tokens=100)

    if not refusal or len(refusal) < 20:
        refusal = (
            f"I'm unable to answer this query. {reason} "
            f"You may want to consult a specialised source."
        )

    return {
        "action"  : "ABSTAIN",
        "response": refusal,
        "reason"  : reason,
    }


# ================================================================
# MAIN PIPELINE
# ================================================================

def run_pipeline(query, known_variables=None, graph_triples=None,
                 missing_variables=None, history=None,
                 kb_text="", verbose=True):

    known_variables   = known_variables   or []
    graph_triples     = graph_triples     or []
    missing_variables = missing_variables or []

    if verbose:
        print(f"\n  Query   : {query}")
        print(f"  Known   : {known_variables}")
        print(f"  Triples : {len(graph_triples)}")

    decision, planner_response = run_planner(
        query, known_variables, graph_triples,
        missing_variables, history
    )

    if verbose:
        print(f"  Decision: {decision}")

    if decision == "ANSWER":
        result = answer_agent(
            query, graph_triples=graph_triples,
            kb_text=kb_text, history=history
        )
    elif decision == "ASK":
        result = ask_agent(
            query, missing_variables, known_variables,
            graph_triples=graph_triples, history=history
        )
    else:
        result = abstain_agent(
            query, known_variables,
            graph_triples=graph_triples,
            missing_variables=missing_variables,
            history=history
        )

    result["planner_reasoning"] = planner_response
    result["graph_triples"]     = graph_triples

    if verbose:
        print(f"  Response: {result['response'][:120]}")

    return result


# ================================================================
# SYNTHETIC TEST CASES
# ================================================================

SYNTHETIC_CASES = [
    # ── ANSWER cases ─────────────────────────────────────────
    {
        "case_id": "SYN_ANS_001", "source": "hotpotqa",
        "turn_type": "single",   "expect": "ANSWER",
        "query"            : "Who founded AcmeCorp and in what year?",
        "kb_text"          : "AcmeCorp was founded by Howard Ellis in 1987 in San Francisco.",
        "graph_triples"    : [
            "howard ellis | found | acmecorp",
            "acmecorp | found in | 1987",
            "acmecorp | locate in | san francisco",
        ],
        "known_variables"  : ["AcmeCorp"],
        "missing_variables": [],
        "history"          : None,
    },
    {
        "case_id": "SYN_ANS_002", "source": "quac",
        "turn_type": "multi",    "expect": "ANSWER",
        "query"            : "What was the revenue in that year?",
        "kb_text"          : "In 2021 AcmeCorp reported annual revenue of $4.2 billion, a 12% increase from the previous year.",
        "graph_triples"    : [
            "acmecorp | report in | 2021",
            "acmecorp | earn | 4.2 billion",
            "4.2 billion | increase from | previous year",
        ],
        "known_variables"  : ["AcmeCorp", "2021"],
        "missing_variables": [],
        "history": [
            {"turn_id": 1, "query": "When did AcmeCorp go public?",
             "action": "ANSWER", "response": "AcmeCorp went public in 2021.",
             "resolved_variable": None}
        ],
    },
    {
        "case_id": "SYN_ANS_003", "source": "sharc",
        "turn_type": "single",   "expect": "ANSWER",
        "query"            : "Can employees request remote work under AcmeCorp policy?",
        "kb_text"          : "AcmeCorp Remote Work Policy (2023): All full-time employees are eligible to request up to 3 days of remote work per week after completing 6 months of employment.",
        "graph_triples"    : [
            "acmecorp policy | allow | remote work",
            "remote work | eligible for | full-time employees",
            "remote work | require | 6 months employment",
            "remote work | limit | 3 days per week",
        ],
        "known_variables"  : ["AcmeCorp", "remote work", "employees"],
        "missing_variables": [],
        "history"          : None,
    },

    # ── ASK cases ─────────────────────────────────────────────
    {
        "case_id": "SYN_ASK_001", "source": "quac",
        "turn_type": "multi",    "expect": "ASK",
        "query"            : "Did she win any awards?",
        "kb_text"          : "Dr. Priya Nair joined AcmeCorp as CTO in 2019. She has received multiple industry recognitions.",
        "graph_triples"    : [
            "priya nair | join | acmecorp",
            "priya nair | serve as | cto",
            "priya nair | requires | ?unknown_1",
        ],
        "known_variables"  : ["awards"],
        "missing_variables": ["who is being referred to"],
        "history": [
            {"turn_id": 1, "query": "Tell me about AcmeCorp leadership.",
             "action": "ANSWER", "response": "AcmeCorp has several key executives.",
             "resolved_variable": None},
        ],
    },
    {
        "case_id": "SYN_ASK_002", "source": "sharc",
        "turn_type": "single",   "expect": "ASK",
        "query"            : "Am I eligible for the AcmeCorp pension plan?",
        "kb_text"          : "AcmeCorp Pension Plan: Eligibility depends on employment type (full-time or part-time), years of service, and age at enrollment.",
        "graph_triples"    : [
            "pension plan | require | employment type",
            "pension plan | require | years of service",
            "pension plan | require | age at enrollment",
            "pension plan | requires | ?unknown_1",
        ],
        "known_variables"  : ["AcmeCorp", "pension plan"],
        "missing_variables": ["employment type", "years of service", "age"],
        "history"          : None,
    },
    {
        "case_id": "SYN_ASK_003", "source": "quac",
        "turn_type": "multi",    "expect": "ASK",
        "query"            : "How did it perform internationally?",
        "kb_text"          : "AcmeCorp's ProductX launched in North America in Q1 2022 and expanded to Europe in Q3 2022.",
        "graph_triples"    : [
            "productx | launch in | north america",
            "productx | expand to | europe",
            "productx | requires | ?unknown_1",
        ],
        "known_variables"  : ["ProductX", "internationally"],
        "missing_variables": ["which time period", "which specific markets"],
        "history": [
            {"turn_id": 1, "query": "What products did AcmeCorp launch in 2022?",
             "action": "ANSWER", "response": "AcmeCorp launched ProductX in 2022.",
             "resolved_variable": None},
            {"turn_id": 2, "query": "When did it launch?",
             "action": "ANSWER", "response": "ProductX launched in Q1 2022.",
             "resolved_variable": None},
        ],
    },

    # ── ABSTAIN cases ─────────────────────────────────────────
    {
        "case_id": "SYN_ABS_001", "source": "hotpotqa",
        "turn_type": "single",   "expect": "ABSTAIN",
        "query"            : "What is the market capitalisation of AcmeCorp's main competitor?",
        "kb_text"          : "AcmeCorp was founded by Howard Ellis in 1987 in San Francisco.",
        "graph_triples"    : [],
        "known_variables"  : ["AcmeCorp", "competitor", "market capitalisation"],
        "missing_variables": [],
        "history"          : None,
    },
    {
        "case_id": "SYN_ABS_002", "source": "contract_nli",
        "turn_type": "single",   "expect": "ABSTAIN",
        "query"            : "Does the AcmeCorp NDA allow reverse engineering of software?",
        "kb_text"          : "AcmeCorp Non-Disclosure Agreement (2023): The Receiving Party agrees to maintain strict confidentiality of all Confidential Information shared by AcmeCorp.",
        "graph_triples"    : [
            "receiving party | maintain | confidentiality",
            "confidential information | share by | acmecorp",
        ],
        "known_variables"  : ["AcmeCorp NDA", "reverse engineering"],
        "missing_variables": ["reverse engineering clause"],
        "history"          : None,
    },
    {
        "case_id": "SYN_ABS_003", "source": "quac",
        "turn_type": "multi",    "expect": "ABSTAIN",
        "query"            : "Is there anything else interesting about this?",
        "kb_text"          : "AcmeCorp was founded by Howard Ellis in 1987 in San Francisco.",
        "graph_triples"    : [],
        "known_variables"  : [],
        "missing_variables": [],
        "history": [
            {"turn_id": 1, "query": "Who founded AcmeCorp?",
             "action": "ANSWER", "response": "Howard Ellis.",
             "resolved_variable": None},
            {"turn_id": 2, "query": "When?",
             "action": "ANSWER", "response": "1987.",
             "resolved_variable": None},
        ],
    },
]


# ================================================================
# RUNNER
# ================================================================

def run_synthetic_verification(cases):
    results = []
    passed  = 0

    print("\n" + "="*65)
    print("  SYNTHETIC PIPELINE VERIFICATION")
    print("  9 cases: 3×ANSWER + 3×ASK + 3×ABSTAIN")
    print("="*65)

    for case in cases:
        print(f"\n{'─'*65}")
        print(f"  Case   : {case['case_id']}  |  "
              f"Source: {case['source'].upper()}  |  "
              f"Turn: {case['turn_type']}")
        print(f"  Expect : {case['expect']}")
        print(f"  Query  : {case['query']}")

        result = run_pipeline(
            query             = case["query"],
            known_variables   = case.get("known_variables",   []),
            graph_triples     = case.get("graph_triples",     []),
            missing_variables = case.get("missing_variables", []),
            history           = case.get("history"),
            kb_text           = case.get("kb_text", ""),
            verbose           = False,
        )

        decision = result["action"]
        correct  = (decision == case["expect"])
        if correct:
            passed += 1

        print(f"  Got    : {decision}  {'✅ PASS' if correct else '❌ FAIL'}")
        print(f"  Response: {result['response'][:150]}")

        results.append({
            "case_id" : case["case_id"],
            "source"  : case["source"],
            "turn"    : case["turn_type"],
            "expect"  : case["expect"],
            "got"     : decision,
            "passed"  : correct,
        })

    print(f"\n{'='*65}")
    print(f"  SUMMARY")
    print(f"{'='*65}")
    print(f"  Total passed : {passed} / {len(cases)}")

    for action in ["ANSWER", "ASK", "ABSTAIN"]:
        subset = [r for r in results if r["expect"] == action]
        n = sum(r["passed"] for r in subset)
        print(f"  {action:10}: {n}/{len(subset)}")

    for turn in ["single", "multi"]:
        subset = [r for r in results if r["turn"] == turn]
        n = sum(r["passed"] for r in subset)
        print(f"  {turn+'-turn':15}: {n}/{len(subset)}")

    print(f"{'='*65}")
    return results


verification_results = run_synthetic_verification(SYNTHETIC_CASES)